# Bangkok Traffy Fondue Data Visualization

This notebook explores the Traffy Fondue dataset, which contains records of civic issues reported by citizens in Bangkok. We will visualize the data to understand the distribution of issues, their locations, and potential correlations with environmental factors like PM2.5 and rainfall.

**Libraries:** `pandas`, `plotly`
**Data Sources:**
- Traffy Fondue Reports
- PM2.5 Data
- Rainfall Data

In [3]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

In [4]:
# Define file paths
traffy_path = "../../data/interim/bangkok_traffy_cleaned.csv"
pm_path =  "../scraping/pm25_data.csv"
department_path =  "../scraping/department_data.csv"
rainfall_path =  "../scraping/rainfall_data.csv"

# Load data
df_traffy = pd.read_csv(traffy_path)
df_pm = pd.read_csv(pm_path)
df_dept = pd.read_csv(department_path)
df_rain = pd.read_csv(rainfall_path)

# Preprocessing: Convert timestamps to datetime
# Using format='mixed' to handle potential inconsistencies in date formats
df_traffy['timestamp'] = pd.to_datetime(df_traffy['timestamp'], format='mixed')
df_traffy['date'] = df_traffy['timestamp'].dt.date
df_traffy['date'] = pd.to_datetime(df_traffy['date'])

df_pm['date'] = pd.to_datetime(df_pm['date'])
df_rain['date'] = pd.to_datetime(df_rain['date'])

print("Data loaded successfully!")
print(f"Traffy shape: {df_traffy.shape}")
print(f"PM2.5 shape: {df_pm.shape}")
print(f"Rainfall shape: {df_rain.shape}")

Data loaded successfully!
Traffy shape: (633341, 21)
PM2.5 shape: (912, 7)
Rainfall shape: (1492, 6)


## 1. Distribution of Issues by Type

Let's examine the most common types of issues reported by citizens. This helps identify the primary concerns in the city.

In [ ]:
# Count issues by type (Top 20)
type_counts = df_traffy['type_clean'].value_counts().head(20).reset_index()
type_counts.columns = ['Issue Type', 'Count']

# Plot
fig = px.bar(type_counts, 
             x='Count', 
             y='Issue Type', 
             orientation='h',
             title='Top 20 Issue Types',
             text='Count',
             color='Count',
             color_continuous_scale='Viridis')

fig.update_layout(yaxis={'categoryorder':'total ascending'}, height=600)
fig.show()

## 2. Issues by District (Top 20)

Which districts have the highest number of reported issues? This visualization highlights the areas that might require more attention.

In [6]:
# Count issues by district
district_counts = df_traffy['district'].value_counts().nlargest(20).reset_index()
district_counts.columns = ['District', 'Count']

# Plot
fig = px.bar(district_counts, 
             x='District', 
             y='Count', 
             title='Top 20 Districts with Most Reported Issues',
             text='Count',
             color='Count',
             color_continuous_scale='Magma')

fig.update_traces(textposition='outside')
fig.show()

## 3. Trend of Reported Issues Over Time

Analyzing the number of issues reported over time can reveal patterns, seasonality, or the impact of specific events.

In [7]:
# Group by date
daily_counts = df_traffy.groupby('date').size().reset_index(name='Count')

# Plot
fig = px.line(daily_counts, 
              x='date', 
              y='Count', 
              title='Daily Number of Reported Issues',
              markers=True)

fig.update_xaxes(rangeslider_visible=True)
fig.show()

## 4. Status of Reported Issues

What is the current state of the reported issues? Are they being resolved?

In [8]:
# Count issues by state
state_counts = df_traffy['state'].value_counts().reset_index()
state_counts.columns = ['State', 'Count']

# Plot
fig = px.pie(state_counts, 
             values='Count', 
             names='State', 
             title='Distribution of Issue Status',
             hole=0.4)

fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

## 5. Geospatial Distribution of Issues

Visualizing the location of issues on a map can help identify hotspots. We will use a sample of the data to avoid overcrowding the map.

In [9]:
# Drop rows with missing coordinates
df_map = df_traffy.dropna(subset=['lat', 'lon'])

# Sample if too large (optional, but good for performance)
if len(df_map) > 5000:
    df_map_sample = df_map.sample(5000)
else:
    df_map_sample = df_map

# Plot
fig = px.scatter_mapbox(df_map_sample, 
                        lat="lat", 
                        lon="lon", 
                        color="type_clean",
                        hover_name="type_clean", 
                        hover_data=["district", "comment"],
                        zoom=10, 
                        height=600,
                        title='Geospatial Distribution of Issues (Sampled)')

fig.update_layout(mapbox_style="open-street-map")
fig.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
fig.show()

## 6. Correlation with PM2.5 Levels

Is there a relationship between the number of reported issues and air quality (PM2.5)? We will merge the datasets by date to investigate.

In [10]:
# Merge data
daily_pm = pd.merge(daily_counts, df_pm, on='date', how='inner')

# Create a dual-axis plot
fig = go.Figure()

fig.add_trace(go.Scatter(x=daily_pm['date'], y=daily_pm['Count'], name="Issue Count", line=dict(color='blue')))
fig.add_trace(go.Scatter(x=daily_pm['date'], y=daily_pm['pm25_avg'], name="PM2.5 Avg", yaxis="y2", line=dict(color='red', dash='dot')))

fig.update_layout(
    title="Daily Issues vs PM2.5 Levels",
    xaxis_title="Date",
    yaxis_title="Number of Issues",
    yaxis2=dict(
        title="PM2.5 (µg/m³)",
        overlaying="y",
        side="right"
    ),
    legend=dict(x=0, y=1.1, orientation='h')
)

fig.show()

# Scatter plot for correlation
fig_corr = px.scatter(daily_pm, x='pm25_avg', y='Count', title="Correlation: PM2.5 vs Issue Count")
fig_corr.show()

## 7. Correlation with Rainfall

Similarly, does rainfall affect the number of issues reported (e.g., flooding, drainage issues)?

In [16]:
# Merge data
daily_rain = pd.merge(daily_counts, df_rain, on='date', how='inner')

# Create a dual-axis plot
fig = go.Figure()

fig.add_trace(go.Scatter(x=daily_rain['date'], y=daily_rain['Count'], name="Issue Count", line=dict(color='blue')))
fig.add_trace(go.Bar(x=daily_rain['date'], y=daily_rain['rainfall_mm'], name="Rainfall (mm)", yaxis="y2", marker_color='black', opacity=1))

fig.update_layout(
    title="Daily Issues vs Rainfall",
    xaxis_title="Date",
    yaxis_title="Number of Issues",
    yaxis2=dict(
        title="Rainfall (mm)",
        overlaying="y",
        side="right"
    ),
    legend=dict(x=0, y=1.1, orientation='h')
)

fig.show()

# Scatter plot for correlation
fig_corr = px.scatter(daily_rain, x='rainfall_mm', y='Count', title="Correlation: Rainfall vs Issue Count")
fig_corr.show()